# <font color=blue> Text Representation </font>

> 1. BOW
> 2. TF-IDF

In [1]:
import spacy

nlp = spacy.load("en_core_web_sm")

In [2]:
corpus = [
    'The planet, Neptune, is the furthest planet from the sun',
    'Jupiter is the largest planet',
    'Mars is the fourth planet from the sun'
]

* Define a function **preprocess**

In [6]:
# Lowercase
corpus = [d.lower() for d in corpus]
print(corpus)

['the planet, neptune, is the furthest planet from the sun', 'jupiter is the largest planet', 'mars is the fourth planet from the sun']


In [3]:
def preprocess(doc):
    doc = nlp(doc)

    preprocessed_tokens = []
    for token in doc:
        if token.is_punct or token.is_stop:
            continue
        else:
            preprocessed_tokens.append(token.lemma_)
    return " ".join(preprocessed_tokens)

In [7]:
# Apply preprocess to each document
preprocessed_corpus = [preprocess(document) for document in corpus]
print(preprocessed_corpus)

['planet neptune furth planet sun', 'jupiter large planet', 'mars fourth planet sun']


## <font color=blue> 1. BOW (aka. Term Frequency) </font>

## <font color=green> Scikit-learn's CountVectorizer Methods </font>

<table>
<tr><td>`fit(raw_documents[, y])`</td><td>Learn a vocabulary dictionary of all tokens in the raw documents.</td></tr>
<tr><td>`transform(raw_documents)`</td><td>Transform documents to document-term matrix.</td></tr>
<tr><td>`fit_transform(raw_documents[, y])`</td><td>Learn the vocabulary dictionary and return document-term matrix.</td></tr>
<tr><td>`get_feature_names_out([input_features])`</td><td>Get output feature names for transformation.</td></tr>
<tr><td>`vocabulary_`</td><td>A dictionary where keys are terms and values are indices in the feature matrix.</td></tr>

</table>

In [4]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer()

In [8]:
# Fit and transform the documents using BOW
corpus_cv = cv.fit_transform(preprocessed_corpus)
corpus_cv

<3x8 sparse matrix of type '<class 'numpy.int64'>'
	with 11 stored elements in Compressed Sparse Row format>

In [9]:
# Get Doc-Term Matrix
print(corpus_cv.toarray())

[[0 1 0 0 0 1 2 1]
 [0 0 1 1 0 0 1 0]
 [1 0 0 0 1 0 1 1]]


In [10]:
# Get feature names
print(cv.get_feature_names_out())

['fourth' 'furth' 'jupiter' 'large' 'mars' 'neptune' 'planet' 'sun']


In [ ]:
# Create Dataframe
import pandas as pd

df_bow = pd.DataFrame(corpus_cv.toarray(), columns = cv.get_feature_names_out())
print(df_bow)

   fourth  furth  jupiter  large  mar  neptune  planet  sun
0       0      1        0      0    0        1       2    1
1       0      0        1      1    0        0       1    0
2       1      0        0      0    1        0       1    1


In [ ]:
df_bow = df_bow.transpose()
print(df_bow)

         0  1  2
fourth   0  0  1
furth    1  0  0
jupiter  0  1  0
large    0  1  0
mar      0  0  1
neptune  1  0  0
planet   2  1  1
sun      1  0  1


In [ ]:
df_bow.columns = ['BOW-Doc{}'.format(i+1) for i in range(len(preprocessed_corpus))]
print(df_bow)

         BOW-Doc1  BOW-Doc2  BOW-Doc3
fourth          0         0         1
furth           1         0         0
jupiter         0         1         0
large           0         1         0
mar             0         0         1
neptune         1         0         0
planet          2         1         1
sun             1         0         1


---
## <font color=blue> 2. TF-IDF </font>

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
vectorizer = TfidfVectorizer()

In [ ]:
corpus_v = vectorizer.fit_transform(preprocessed_corpus)
corpus_v

<3x8 sparse matrix of type '<class 'numpy.float64'>'
	with 11 stored elements in Compressed Sparse Row format>

In [ ]:
tfidf_score = corpus_v.toarray()
print(tfidf_score)

[[0.         0.50165133 0.         0.         0.         0.50165133
  0.59256672 0.38151877]
 [0.         0.         0.65249088 0.65249088 0.         0.
  0.38537163 0.        ]
 [0.5844829  0.         0.         0.         0.5844829  0.
  0.34520502 0.44451431]]


In [ ]:
# Create Dataframe
import pandas as pd

df_tfidf = pd.DataFrame(tfidf_score, columns = vectorizer.get_feature_names_out())
print(df_tfidf)

     fourth     furth   jupiter     large       mar   neptune    planet  \
0  0.000000  0.501651  0.000000  0.000000  0.000000  0.501651  0.592567   
1  0.000000  0.000000  0.652491  0.652491  0.000000  0.000000  0.385372   
2  0.584483  0.000000  0.000000  0.000000  0.584483  0.000000  0.345205   

        sun  
0  0.381519  
1  0.000000  
2  0.444514  


In [ ]:
df_tfidf = df_tfidf.transpose()
df_tfidf.columns = ['Tfidf-Doc{}'.format(i+1) for i in range(len(preprocessed_corpus))]
print(df_tfidf)

         Tfidf-Doc1  Tfidf-Doc2  Tfidf-Doc3
fourth     0.000000    0.000000    0.584483
furth      0.501651    0.000000    0.000000
jupiter    0.000000    0.652491    0.000000
large      0.000000    0.652491    0.000000
mar        0.000000    0.000000    0.584483
neptune    0.501651    0.000000    0.000000
planet     0.592567    0.385372    0.345205
sun        0.381519    0.000000    0.444514


In [ ]:
# Merge Dataframes for comparison
df_merged = df_bow.join(df_tfidf)
print(df_merged)

         BOW-Doc1  BOW-Doc2  BOW-Doc3  Tfidf-Doc1  Tfidf-Doc2  Tfidf-Doc3
fourth          0         0         1    0.000000    0.000000    0.584483
furth           1         0         0    0.501651    0.000000    0.000000
jupiter         0         1         0    0.000000    0.652491    0.000000
large           0         1         0    0.000000    0.652491    0.000000
mar             0         0         1    0.000000    0.000000    0.584483
neptune         1         0         0    0.501651    0.000000    0.000000
planet          2         1         1    0.592567    0.385372    0.345205
sun             1         0         1    0.381519    0.000000    0.444514


---
## 2. Inverse Document Frequency (IDF)

> * Document Frequency (DF)
> * IDF

In [ ]:
#from sklearn.feature_extraction.text import TfidfVectorizer

#vectorizer = TfidfVectorizer()

In [ ]:
#corpus_v = vectorizer.fit_transform(preprocessed_corpus)
#corpus_v

In [ ]:
#Key: term, Value: index number of term
vectorizer.vocabulary_

{'planet': 6,
 'neptune': 5,
 'furth': 1,
 'sun': 7,
 'jupiter': 2,
 'large': 3,
 'mar': 4,
 'fourth': 0}

In [ ]:
# Get index number of term
vectorizer.vocabulary_.get('planet')

6

In [ ]:
# Get idf score of term
vectorizer.idf_[6]

1.0

In [ ]:
# Get idf score of terms
terms = vectorizer.get_feature_names_out()

for term in terms:
    index = vectorizer.vocabulary_.get(term)
    print(f'{term:{10}} {vectorizer.idf_[index]}')

fourth     1.6931471805599454
furth      1.6931471805599454
jupiter    1.6931471805599454
large      1.6931471805599454
mar        1.6931471805599454
neptune    1.6931471805599454
planet     1.0
sun        1.2876820724517808
